# Regresión de potencia eléctrica: selección del modelo

Consulta la [metodología](../docs/metodologia.md) y ejecuta las etapas en orden. Los comentarios se calculan con los resultados guardados.

In [1]:
from pathlib import Path
import sys
here = Path.cwd().resolve()
def _buscar_proyecto(paquete):
    """Encuentra el paquete propio desde el proyecto, tema o portafolio."""
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (base / "src" / paquete).is_dir():
            return base
        for container in (base, base / "Analisis"):
            for pattern in (f"*/src/{paquete}", f"*/*/src/{paquete}"):
                for source in sorted(container.glob(pattern)):
                    parts = source.relative_to(container).parts
                    if "back" in parts or any(p.startswith(".") for p in parts):
                        continue
                    candidate = source.parent.parent
                    if (candidate / "pyproject.toml").is_file():
                        return candidate
    return None

PROJECT_ROOT = _buscar_proyecto("power_plant_regression")
if PROJECT_ROOT is None:
    raise RuntimeError("Abre este notebook desde su proyecto independiente.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from power_plant_regression.alerts import configurar_notebook, mostrar_alertas
from IPython.display import display, Markdown, Image
configurar_notebook()


In [2]:
from power_plant_regression.analysis import run_stage
import pandas as pd
metrics,comment=run_stage(PROJECT_ROOT,"modelado")
display(Markdown(comment))
scalars={k:v for k,v in metrics.items() if not isinstance(v,(dict,list))}
if scalars:display(pd.DataFrame(list(scalars.items()),columns=["Indicador","Valor"]))
if "candidatos" in metrics:display(pd.DataFrame(metrics["candidatos"]))
for filename in ["eda.png","evaluacion.png"]:
    path=PROJECT_ROOT/"reports/modelado"/filename
    if path.exists():display(Image(filename=str(path)))
for filename in ["estimaciones.csv","metricas_test.csv","horas_por_empleado.csv"]:
    path=PROJECT_ROOT/"reports/modelado"/filename
    if path.exists():display(pd.read_csv(path).head(20))

# Regresión de potencia eléctrica: modelado

Se elige **HGB_hojas_31** por MAE de validación, entre Ridge y boosting. El modelo se ajusta exclusivamente en train y no se reentrena con validación. Una partición de calibración separada de **953 casos** fija un radio conformal de **±4.942 MW**, con cobertura nominal **90%**. Test permanece reservado. La cobertura marginal requiere intercambiabilidad; no garantiza cobertura condicional ni temporal.


,Indicador,Valor
0,criterio,MAE de validación
1,modelo_elegido,HGB_hojas_31
2,calibracion_filas,953
3,radio_intervalo_mw,4.942154
4,cobertura_nominal,0.9


,modelo,casos,mae_mw,rmse_mw,r2
0,Ridge_alpha_1.0,952,3.718398,4.821370,0.924040
1,Ridge_alpha_10.0,952,3.720855,4.821585,0.924033
2,Ridge_alpha_100.0,952,3.753621,4.847012,0.923229
3,HGB_hojas_15,952,2.697084,3.970914,0.948474
4,HGB_hojas_31,952,2.489533,3.789612,0.953072


In [3]:
mostrar_alertas(PROJECT_ROOT / "reports/alertas/04_modelado.json")
from power_plant_regression.informe import generar_informe
generar_informe(PROJECT_ROOT)
display(Markdown("[Abrir informe integral](../reports/informe_final/informe_final.md)"))

[Abrir informe integral](../reports/informe_final/informe_final.md)